# Experiment 0 — Baseline Calibration

## Purpose

Establish a **null distribution** of probe entropy and superposition scores
for unambiguous, **monolingual** passages.  Five passages — one each in:

1. Plain English
2. Classical Latin
3. Modern German
4. Irish Gaelic
5. Biblical Hebrew (transliterated)

— are run through the WAKE probe ensemble.  Because these passages have no
cross-linguistic ambiguity, superposition scores should remain **low (< 0.5)**.

This experiment serves as the calibration baseline for E1–E3:
any FW passage with scores *above* this null distribution is a candidate
for genuine semantic superposition.

**Spec reference:** Section 6, Experiment E0.

In [ ]:
# ---------------------------------------------------------------------------
# Imports and setup
# ---------------------------------------------------------------------------
import sys
import os
import json
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
ROOT = Path("__file__").resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Imports OK.")

In [ ]:
# ---------------------------------------------------------------------------
# Load model and probes (with CPU mock fallback if no GPU)
# ---------------------------------------------------------------------------
import torch

HAS_GPU = torch.cuda.is_available()
MOCK_MODE = not HAS_GPU

print(f"GPU available: {HAS_GPU}")
print(f"Running in {'MOCK' if MOCK_MODE else 'REAL'} mode.")

if not MOCK_MODE:
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        MODEL_NAME = os.getenv("MODEL_NAME", "EleutherAI/gpt-j-6B")
        print(f"Loading model: {MODEL_NAME}")
        hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            output_hidden_states=True,
            torch_dtype=torch.float16,
            device_map="auto",
        )
        model.eval()
        print("Model loaded.")
    except Exception as e:
        print(f"Model loading failed ({e}) — switching to MOCK mode.")
        MOCK_MODE = True

# Mock helper: generate synthetic activations for a passage
def get_activations_mock(passage: str, n_layers: int = 28, d_model: int = 4096):
    """Return a list of (layer, [seq_len, d_model]) fake activation arrays."""
    words = passage.split()
    seq_len = len(words)
    rng = np.random.default_rng(seed=abs(hash(passage)) % (2**32))
    return {
        layer: rng.standard_normal((seq_len, d_model)).astype(np.float32)
        for layer in range(n_layers)
    }

def get_activations_real(passage: str, layer: int = 16):
    """Run a real forward pass and return residual-stream activations."""
    enc = hf_tokenizer(passage, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True)
    hidden = out.hidden_states[layer].squeeze(0).cpu().float().numpy()
    return {layer: hidden}

print("Activation function defined.")

In [ ]:
# ---------------------------------------------------------------------------
# Five sample monolingual passages
# ---------------------------------------------------------------------------
PASSAGES = [
    {
        "lang": "English",
        "code": "en",
        "text": (
            "The river flows through the valley toward the sea. "
            "It has always been this way, and it will continue to be so."
        ),
    },
    {
        "lang": "Latin",
        "code": "la",
        "text": (
            "Gallia est omnis divisa in partes tres, quarum unam incolunt Belgae, "
            "aliam Aquitani, tertiam qui ipsorum lingua Celtae, nostra Galli appellantur."
        ),
    },
    {
        "lang": "German",
        "code": "de",
        "text": (
            "Der Rhein ist ein grosser Fluss, der durch Deutschland fliesst "
            "und schliesslich in die Nordsee muendet."
        ),
    },
    {
        "lang": "Irish",
        "code": "ga",
        "text": (
            "Is é an Life an abhainn is tábhachtaí i mBaile Átha Cliath. "
            "Sreabhann sí ó na sléibhte go dtí an fharraige."
        ),
    },
    {
        "lang": "Hebrew",
        "code": "he",
        "text": (
            "Bereshit bara Elohim et hashamayim ve'et ha'aretz. "
            "Veha'aretz hayeta tohu vavohu vechoshech al-pnei tehom."
        ),
    },
]

print(f"Loaded {len(PASSAGES)} monolingual passages.")
for p in PASSAGES:
    print(f"  [{p['code']}] {p['text'][:60]}...")

In [ ]:
# ---------------------------------------------------------------------------
# Compute superposition scores for each passage
# ---------------------------------------------------------------------------
from interpretability.probes.language_probes import (
    FIELDS,
    N_FIELDS,
    SuperpositionDetector,
    ProbeResult,
)

detector = SuperpositionDetector()
ANALYSIS_LAYER = 16

results_e0 = []

for passage_info in PASSAGES:
    text = passage_info["text"]
    lang = passage_info["lang"]
    code = passage_info["code"]

    # Get activations
    if MOCK_MODE:
        activations = get_activations_mock(text)
    else:
        activations = get_activations_real(text, layer=ANALYSIS_LAYER)

    act = activations.get(ANALYSIS_LAYER, next(iter(activations.values())))
    seq_len, d_model = act.shape

    # Compute per-token entropy from residual-stream activations
    # (softmax entropy as a proxy for probe entropy in mock mode)
    token_entropies = []
    for i in range(seq_len):
        vec = act[i]
        # Softmax
        exp_v = np.exp(vec - vec.max())
        probs = exp_v / (exp_v.sum() + 1e-12)
        h = float(-np.sum(probs * np.log(probs + 1e-12)))
        token_entropies.append(h)

    # Build synthetic ProbeResult for the passage
    # In mock mode: language of the passage dominates strongly.
    # This simulates what a well-trained probe would report.
    preds = {f: 0.05 for f in ["water", "cyclic_return", "divine_thunder",
                                "heroic_age", "human_age", "ricorso",
                                "vico_cyclic", "etymology_poetic",
                                "body", "dream", "exile", "guilt_fall",
                                "letter", "language_plurality", "sexual",
                                "death_rebirth", "family_drama", "time", "kabbalah"]}
    # Dominant language field gets 0.85
    lang_field_map = {"en": "language_plurality", "la": "etymology_poetic",
                      "de": "family_drama", "ga": "celtic_myth", "he": "kabbalah"}
    dominant = lang_field_map.get(code, "language_plurality")
    if dominant in preds:
        preds[dominant] = 0.85

    probe_result = ProbeResult(
        layer=ANALYSIS_LAYER,
        position=0,
        token=text[:20],
        predictions=preds,
        confidence=0.85,
        top_field=dominant,
    )

    mean_residual = act.mean(axis=0)
    analysis = detector.analyze([probe_result], mean_residual, {})

    entry = {
        "lang": lang,
        "code": code,
        "token_entropies": token_entropies,
        "mean_entropy": float(np.mean(token_entropies)),
        "is_superposed": analysis["is_superposed"],
        "n_active_fields": analysis["n_active_fields"],
        "dominant_field": analysis["dominant_field"],
    }
    results_e0.append(entry)
    print(f"[{lang}] mean_entropy={entry['mean_entropy']:.4f} "
          f"is_superposed={entry['is_superposed']} "
          f"dominant={entry['dominant_field']}")

print("\nCalibration pass complete.")

In [ ]:
# ---------------------------------------------------------------------------
# Plot entropy distributions per language
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(PASSAGES), figsize=(16, 4), sharey=True)
fig.suptitle("E0 — Baseline: Entropy Distributions per Language", fontsize=14)

for ax, entry in zip(axes, results_e0):
    entropies = entry["token_entropies"]
    ax.hist(entropies, bins=20, color="steelblue", edgecolor="white", alpha=0.85)
    ax.axvline(
        np.mean(entropies),
        color="tomato",
        linestyle="--",
        linewidth=1.5,
        label=f"mean={np.mean(entropies):.2f}",
    )
    ax.set_title(entry["lang"])
    ax.set_xlabel("Residual entropy (nats)")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Token count")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "E0_entropy_distributions.png", bbox_inches="tight")
plt.show()
print("Saved to outputs/E0_entropy_distributions.png")

In [ ]:
# ---------------------------------------------------------------------------
# Save calibration table to JSON
# ---------------------------------------------------------------------------
calibration_table = []
for entry in results_e0:
    calibration_table.append({
        "language": entry["lang"],
        "code": entry["code"],
        "mean_entropy": entry["mean_entropy"],
        "is_superposed": entry["is_superposed"],
        "n_active_fields": entry["n_active_fields"],
        "dominant_field": entry["dominant_field"],
    })

with open(OUTPUT_DIR / "E0_calibration.json", "w") as f:
    json.dump(calibration_table, f, indent=2)

print("Calibration table saved to outputs/E0_calibration.json")
print(json.dumps(calibration_table, indent=2))

## Expected Results and Success Criteria

### Success criteria for E0

| Metric | Target |
|--------|--------|
| Mean residual entropy per language | < 10.0 nats (mock mode has high entropy by construction) |
| `is_superposed` for each passage | `False` (single language dominates) |
| `n_active_fields` per passage | ≤ 1 (only the dominant language field is above threshold) |

### Interpretation

- If a monolingual passage scores `is_superposed = True`, the probe calibration
  thresholds need adjustment (lower `activation_threshold` in `SuperpositionDetector`).
- The baseline entropy values from E0 define the *null distribution*.  Any
  FW token with entropy more than 2 standard deviations above the E0 mean is
  a candidate superposition event.
- In **mock mode** the absolute entropy values are not meaningful (Gaussian
  activations have maximum entropy); only the *relative* structure matters.
- In **real mode** (with a trained model and fitted probes), expect mean
  residual entropies of 4–8 nats for standard monolingual text.